In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from torch.distributions import Normal, Independent

import sys
sys.path.append('../../drnpe')
sys.path.append('.')

import lightning

from hydra import compose, initialize
from hydra.utils import instantiate

import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

We load the configs and instantiate the data module:

In [ ]:
with initialize(version_base=None, config_path="conf"):
    cfg_npe = compose(config_name="config_cs_npe")
    cfg_drnpe = compose(config_name="config_cs_drnpe")
    cfg_npe_flow = compose(config_name="config_cs_npe_flow")
    cfg_drnpe_flow = compose(config_name="config_cs_drnpe_flow")

lightning.seed_everything(cfg_npe.seed)

datamodule = instantiate(cfg_npe.datamodule)

## Gaussian Variational Distribution Encoders

We load encoders trained with a Gaussian variational distribution:

In [ ]:
# NPE with Gaussian variational distribution
encoder_npe = instantiate(cfg_npe.encoder)
state_dict = torch.load("../../trained_ckpts/network_cs_npe.ckpt", map_location=device)['state_dict']
encoder_npe.load_state_dict(state_dict)
encoder_npe = encoder_npe.to(device).eval()

# DRNPE with Gaussian variational distribution
encoder_drnpe = instantiate(cfg_drnpe.encoder)
state_dict = torch.load("../../trained_ckpts/network_cs_drnpe.ckpt", map_location=device)['state_dict']
encoder_drnpe.load_state_dict(state_dict)
encoder_drnpe = encoder_drnpe.to(device).eval()

print("Loaded Gaussian encoders")

## Neural Spline Flow Encoders

We load encoders trained with neural spline flow variational distributions:

In [ ]:
# NPE with neural spline flow
encoder_npe_flow = instantiate(cfg_npe_flow.encoder)
state_dict = torch.load("../../trained_ckpts/network_cs_npe_flow.ckpt", map_location=device)['state_dict']
encoder_npe_flow.load_state_dict(state_dict)
encoder_npe_flow = encoder_npe_flow.to(device).eval()

# DRNPE with neural spline flow
encoder_drnpe_flow = instantiate(cfg_drnpe_flow.encoder)
state_dict = torch.load("../../trained_ckpts/network_cs_drnpe_flow.ckpt", map_location=device)['state_dict']
encoder_drnpe_flow.load_state_dict(state_dict)
encoder_drnpe_flow = encoder_drnpe_flow.to(device).eval()

print("Loaded flow encoders")

## Helper Functions

**Misspecification type:**
- `"necrosis"`: Remove cancer cells in core regions (within 0.8r_i of parent) with probability `misspec_param` per parent

In [ ]:
def generate_test_data(num_samples, misspec_type=None, misspec_param=None):
    """Generate well-specified or misspecified test data.

    Returns:
        z: True parameter values [num_samples, 3] on device
        x: Summary statistics [num_samples, 4] on device
    """
    if misspec_type is None:
        dataset = datamodule.generate_dataset(num_samples)
    else:
        dataset = datamodule.generate_misspecified_data(num_samples, misspec_type, misspec_param)
    z = dataset.tensors[0].to(device)
    x = dataset.tensors[1].to(device)
    return z, x

In [ ]:
param_names = [r'$\lambda_c$', r'$\lambda_p$', r'$\lambda_d$']

def plot_coverage_probabilities(num_test_samples, encoder, misspec_configs, title=""):
    """Coverage plot for Gaussian encoders, marginal per parameter dimension."""
    num_subplots = len(misspec_configs)
    fig, axes = plt.subplots(3, num_subplots, figsize=(4 * num_subplots, 12))
    if num_subplots == 1:
        axes = axes[:, None]

    for j, (label, misspec_type, misspec_param) in enumerate(misspec_configs):
        z, x = generate_test_data(num_test_samples, misspec_type, misspec_param)

        with torch.inference_mode():
            mu, logsigma = encoder.net(x)  # [batch, 3]
            sigma = torch.exp(logsigma)

        confidence_levels = torch.linspace(0.05, 0.95, steps=19)
        quantiles = Normal(0, 1).icdf(1 - (1 - confidence_levels) / 2).to(device)

        for dim in range(3):
            mu_d = mu[:, dim]
            sigma_d = sigma[:, dim]
            z_d = z[:, dim]

            lower = mu_d.unsqueeze(-1) - quantiles * sigma_d.unsqueeze(-1)
            upper = mu_d.unsqueeze(-1) + quantiles * sigma_d.unsqueeze(-1)

            coverage = ((z_d.unsqueeze(-1) >= lower) & (z_d.unsqueeze(-1) <= upper)).float().mean(0)

            axes[dim, j].scatter(confidence_levels.cpu(), coverage.cpu(), s=15)
            axes[dim, j].axline((0, 0), slope=1, color='black')
            axes[dim, j].set_xlim(0, 1)
            axes[dim, j].set_ylim(0, 1)
            axes[dim, j].set_title(f'{label}')
            axes[dim, j].set_xlabel('Nominal coverage')
            axes[dim, j].set_ylabel(f'Empirical coverage ({param_names[dim]})')

    if title:
        fig.suptitle(title, y=1.02)
    fig.tight_layout()

In [ ]:
def plot_coverage_probabilities_flow(num_test_samples, encoder, misspec_configs, num_posterior_samples=1000, title=""):
    """Coverage plot for flow-based encoders using Monte Carlo sampling."""
    num_subplots = len(misspec_configs)
    fig, axes = plt.subplots(3, num_subplots, figsize=(4 * num_subplots, 12))
    if num_subplots == 1:
        axes = axes[:, None]

    for j, (label, misspec_type, misspec_param) in enumerate(misspec_configs):
        z, x = generate_test_data(num_test_samples, misspec_type, misspec_param)

        with torch.inference_mode():
            posterior_samples = encoder.flow.sample(num_posterior_samples, x)  # [batch, num_samples, 3]

        confidence_levels = torch.linspace(0.05, 0.95, steps=19)
        lower_quantiles = (1 - confidence_levels) / 2
        upper_quantiles = 1 - lower_quantiles

        for dim in range(3):
            samples_d = posterior_samples[:, :, dim]  # [batch, num_samples]
            z_d = z[:, dim]

            lower = torch.quantile(samples_d, lower_quantiles.to(device), dim=1).T
            upper = torch.quantile(samples_d, upper_quantiles.to(device), dim=1).T

            coverage = ((z_d.unsqueeze(-1) >= lower) & (z_d.unsqueeze(-1) <= upper)).float().mean(0)

            axes[dim, j].scatter(confidence_levels.cpu(), coverage.cpu(), s=15)
            axes[dim, j].axline((0, 0), slope=1, color='black')
            axes[dim, j].set_xlim(0, 1)
            axes[dim, j].set_ylim(0, 1)
            axes[dim, j].set_title(f'{label}')
            axes[dim, j].set_xlabel('Nominal coverage')
            axes[dim, j].set_ylabel(f'Empirical coverage ({param_names[dim]})')

    if title:
        fig.suptitle(title, y=1.02)
    fig.tight_layout()

## Results: Well-Specified and Necrosis Misspecification

### NPE (Gaussian)

In [ ]:
misspec_configs = [
    ('Well-specified', None, None),
    ('Necrosis (p=0.5)', 'necrosis', 0.5),
    ('Necrosis (p=0.75)', 'necrosis', 0.75),
    ('Necrosis (p=1.0)', 'necrosis', 1.0),
]
plot_coverage_probabilities(1000, encoder_npe, misspec_configs, title="NPE (Gaussian)")

### DRNPE (Gaussian)

In [ ]:
plot_coverage_probabilities(1000, encoder_drnpe, misspec_configs, title="DRNPE (Gaussian)")

### NPE (Flow)

In [ ]:
plot_coverage_probabilities_flow(1000, encoder_npe_flow, misspec_configs, title="NPE (Flow)")

### DRNPE (Flow)

In [ ]:
plot_coverage_probabilities_flow(1000, encoder_drnpe_flow, misspec_configs, title="DRNPE (Flow)")

## 2D Posterior Plots

We visualize 2D posterior scatter plots for selected test observations under necrosis misspecification:

In [ ]:
z_test, x_test = generate_test_data(5, misspec_type='necrosis', misspec_param=0.75)

num_posterior_samples = 2000

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i in range(5):
    x_i = x_test[i:i+1]
    z_true = z_test[i].cpu()

    with torch.inference_mode():
        samples_npe = encoder_npe_flow.flow.sample(num_posterior_samples, x_i).squeeze(0).cpu()
        samples_drnpe = encoder_drnpe_flow.flow.sample(num_posterior_samples, x_i).squeeze(0).cpu()

    for row, (samples, label) in enumerate([(samples_npe, 'NPE Flow'), (samples_drnpe, 'DRNPE Flow')]):
        axes[row, i].scatter(samples[:, 0], samples[:, 1], alpha=0.1, s=5)
        axes[row, i].scatter(z_true[0], z_true[1], color='red', s=50, zorder=5, marker='x', linewidths=2)
        axes[row, i].set_xlabel(r'$\lambda_c$')
        axes[row, i].set_ylabel(r'$\lambda_p$')
        axes[row, i].set_title(f'{label} - obs {i+1}')

fig.suptitle('2D Posteriors under Necrosis (p=0.75)', y=1.02)
fig.tight_layout()

## Log Variational Densities

We compare the log variational density $\log q(z|x)$ across all four encoders under necrosis misspecification:

In [ ]:
num_test_samples = 1000
z_test, x_test = generate_test_data(num_test_samples, misspec_type='necrosis', misspec_param=0.75)

with torch.inference_mode():
    mu_npe, logsigma_npe = encoder_npe.net(x_test)
    logq_npe = Independent(Normal(mu_npe, torch.exp(logsigma_npe)), 1).log_prob(z_test)

    mu_drnpe, logsigma_drnpe = encoder_drnpe.net(x_test)
    logq_drnpe = Independent(Normal(mu_drnpe, torch.exp(logsigma_drnpe)), 1).log_prob(z_test)

    logq_npe_flow = encoder_npe_flow.flow.log_prob(z_test, x_test)
    logq_drnpe_flow = encoder_drnpe_flow.flow.log_prob(z_test, x_test)

fig, ax = plt.subplots(figsize=(8, 5))
data = [
    logq_npe.cpu().numpy(),
    logq_drnpe.cpu().numpy(),
    logq_npe_flow.cpu().numpy(),
    logq_drnpe_flow.cpu().numpy()
]
labels = ['NPE\n(Gaussian)', 'DRNPE\n(Gaussian)', 'NPE\n(Flow)', 'DRNPE\n(Flow)']

ax.boxplot(data, tick_labels=labels, showfliers=False)
ax.set_ylabel('$\\log q(z|x)$')
ax.set_title('Log Variational Density (necrosis p=0.75)')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
fig.tight_layout()